# AirBnB Product Project

* By: Nhi Bui

# Statistical Testing for Factor Impact on Review Ratings

## Question 1
What host listing factors controlled by hosts most significantly impact review ratings?

## My proposed approach
Validate EDA findings with statistical tests to determine 
if the observed differences in review ratings across host-controlled 
factors are statistically significant.

In [23]:
#Reload data
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/Airbnb_Open_Data_Cleaned.csv')

# ANOVA Test - Room Type

In [24]:
#Split data into groups by room type
df['room type'].unique()

<StringArray>
['Private room', 'Entire home/apt', 'Shared room', 'Hotel room']
Length: 4, dtype: str

In [25]:
# Create a variable for each group's review ratings
private = df[df['room type'] == 'Private room']['review rate number']
entire = df[df['room type'] == 'Entire home/apt']['review rate number']
shared = df[df['room type'] == 'Shared room']['review rate number']
hotel = df[df['room type'] == 'Hotel room']['review rate number']

In [26]:
#Run ANOVA test to compare the means of the review ratings across the different room types
f_stat, p_value = stats.f_oneway(private, entire, shared, hotel)
print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_value:.4f}")

F-statistic: 3.2804
P-value: 0.0200


Because p-value = 0.02 < 0.05
=> There is a statistically significant difference in review ratings between room types
=> Reject the NULL hypothesis
=> This confirms our earlier statement of review rate number affected by different room types

**Post-hoc test**

To find out which specific room types differ from each other, you need a post-hoc test using Tukey's HSD 

In [27]:
#Check for Turkey HSD post-hoc test to see which groups differ
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey = pairwise_tukeyhsd(df['review rate number'], df['room type'], alpha=0.05)
print(tukey)

       Multiple Comparison of Means - Tukey HSD, FWER=0.05        
     group1        group2    meandiff p-adj   lower  upper  reject
------------------------------------------------------------------
Entire home/apt   Hotel room    0.265 0.1932 -0.0778 0.6079  False
Entire home/apt Private room    0.025 0.0724 -0.0015 0.0514  False
Entire home/apt  Shared room   0.0321  0.801 -0.0588 0.1229  False
     Hotel room Private room  -0.2401 0.2739  -0.583 0.1028  False
     Hotel room  Shared room   -0.233  0.328 -0.5867 0.1208  False
   Private room  Shared room   0.0071 0.9971 -0.0841 0.0983  False
------------------------------------------------------------------


**Conclusion**
* ANOVA confirms a statistically significant difference exists across room types (p = 0.02)
* However, Tukey's post-hoc test shows no single pair is significantly different after correction
* The strongest contrast is between Entire home/apt and Private room (p = 0.0724)
=> This suggests the effect is distributed across groups rather than driven by one standout room type

**Recommendation**

Because the differences in ratings between room types are real but small to justify treating one room type differently from others, Airbnb should focus on improving quality within each room type (amenity checklists, service quality guidelines, professional photo programs, and responsive communication tools) instead of trying to push hosts toward a specific room type 

# ANOVA Test - Price

In [28]:
#Check bins of price
df['price'].dtype

dtype('float64')

Because type of price is float, we need to recreate the bins

In [29]:
#Recreate bins for price
df['price_group'] = pd.cut(df['price'], 
                           bins=[0, 50, 500, 1000, 1500], 
                           labels=['Low', 'Medium', 'High', 'Very High'])

In [30]:
# Create a variable for each group's review ratings
low = df[df['price_group'] == 'Low']['review rate number']
medium = df[df['price_group'] == 'Medium']['review rate number']
high = df[df['price_group'] == 'High']['review rate number']
very_high = df[df['price_group'] == 'Very High']['review rate number']

In [31]:
#Run ANOVA test to compare the means of the review ratings across the different price groups
f_stat, p_value = stats.f_oneway(low, medium, high, very_high)
print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_value:.4f}")

F-statistic: 5.5889
P-value: 0.0008


Because p-value = 0.0008 < 0.05
=> There is a statistically significant difference in review ratings between prices
=> Reject the NULL hypothesis
=> This confirms our earlier statement of review rate number affected by different prices

**Post-hoc test**

To find out which specific price levels differ from each other, you need a post-hoc test using Tukey's HSD 

In [32]:
#Check for Turkey HSD post-hoc test to see which groups differ
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey = pairwise_tukeyhsd(df['review rate number'], df['price_group'], alpha=0.05)
print(tukey)

  Multiple Comparison of Means - Tukey HSD, FWER=0.05  
group1   group2  meandiff p-adj   lower   upper  reject
-------------------------------------------------------
  High       Low  -0.1429 0.7972 -0.5446  0.2587  False
  High    Medium   0.0153 0.5159 -0.0134   0.044  False
  High Very High  -0.0427 0.0153 -0.0796 -0.0059   True
   Low    Medium   0.1583 0.7422 -0.2434    0.56  False
   Low Very High   0.1002 0.9191 -0.3022  0.5026  False
Medium Very High  -0.0581 0.0004 -0.0955 -0.0206   True
-------------------------------------------------------


**Conclusion**
* Two pairs show reject = True:
    * High vs Very High (p = 0.0153) => significant difference
    * Medium vs Very High (p = 0.0004) => highly significant difference
=> Both involve Very High price being significantly lower in ratings. => This  confirms our earlier EDA hypothesis about the expectation gap
=> Very high priced listings genuinely underperform compared to medium and high priced ones

* Meanwhile, Low price level vs everything shows reject = False
=> The low price effect we saw in EDA wasn't strong enough to be statistically significant


**Recommendation**

1. Pricing guidance for premium hosts
We should implement a pricing quality check that alerts hosts when their price significantly exceeds the neighbourhood average without matching amenity or quality benchmarks

2. Guest expectation management 
For very high priced listings, Airbnb could require more detailed listing descriptions, verified amenity lists, and professional photos to ensure the listing delivers on its price promise

3. Pricing recommendation tool 
Provide hosts with data-driven pricing recommendations that optimize for both revenue and guest satisfaction
=> Guide them toward the medium-to-high range where ratings peak

Given the smaller effect sizes observed in EDA (0.08 points), statistical testing may not needed for availability and minimum nights as they are unlikely to reach significance

However, for reducing risks, we will still conduct tests for these 2 factors

# ANOVA Test - Availability

In [33]:
#Check bins for availability
df['availability 365'].dtype

dtype('float64')

In [34]:
#Recreate bins for availability
df['availability_group'] = pd.cut(df['availability 365'], 
                                   bins=[0, 90, 180, 270, 365], 
                                   labels=['Low', 'Medium', 'High', 'Always Available'])

In [35]:
# Create a variable for each group's review ratings
a_low = df[df['availability_group'] == 'Low']['review rate number']
a_medium = df[df['availability_group'] == 'Medium']['review rate number']
a_high = df[df['availability_group'] == 'High']['review rate number']
a_very_high = df[df['availability_group'] == 'Always Available']['review rate number']

In [36]:
#Run ANOVA test to compare the means of the review ratings across the different availability groups
f_stat, p_value = stats.f_oneway(a_low, a_medium, a_high, a_very_high)
print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_value:.4f}")

F-statistic: 8.6482
P-value: 0.0000


Since p < 0.00001, this is the most significant result across all factors

=> Despite the small 0.08 gap in EDA, with 63K+ data points even small differences can be statistically significant 
=> Need to test Turkey HSD to actually test this difference

**Post-hoc test**

In [37]:
#Check the Turkey HSD post-hoc test to see which groups differ
tukey_avail = pairwise_tukeyhsd(df['review rate number'], df['availability_group'], alpha=0.05)
print(tukey_avail)

     Multiple Comparison of Means - Tukey HSD, FWER=0.05      
     group1      group2 meandiff p-adj   lower   upper  reject
--------------------------------------------------------------
Always Available   High  -0.0493   0.01 -0.0899 -0.0086   True
Always Available    Low   0.0263 0.1649 -0.0064  0.0591  False
Always Available Medium  -0.0127 0.8234 -0.0506  0.0251  False
            High    Low   0.0756    0.0  0.0364  0.1148   True
            High Medium   0.0365 0.1364  -0.007  0.0801  False
             Low Medium  -0.0391 0.0291 -0.0753 -0.0028   True
--------------------------------------------------------------


* Tukey's HSD: Three significant pairs identified
  * High vs Low (p = 0.00) 
  => Low availability rates significantly higher
  * Always Available vs High (p = 0.01)
  => Always Available outperforms High
  * Low vs Medium (p = 0.03) 
  => Low availability rates significantly higher

=> Availability is the  strongest predictor of review ratings, confirming the hypothesis that hosts with lower availability are more selective and invest more in guest experience quality

=> High availability (181-270 days) consistently underperforms, suggesting that year-round availability without maintenance breaks leads to quality deterioration

**Product Recommendation:**
1. Encourage hosts to build in maintenance periods rather than listing year-round
2. Introduce a "quality rest" feature that helps hosts schedule downtime between bookings
3. Track consistently high-availability listings with declining ratings for proactive quality intervention

# ANOVA Test - Minimum nights

In [38]:
#Check bins for minimum nights
df['minimum nights'].dtype

dtype('float64')

In [ ]:
#Because we can't run the bins, we have to check category again
df['minimum nights'].head()

0    Monthly stay
1    Monthly stay
2    Monthly stay
3       Long term
4      Short stay
Name: minimum nights, dtype: category
Categories (4, str): ['Short stay' < 'Medium stay' < 'Monthly stay' < 'Long term']

In [22]:
df['minimum nights'].isnull().sum()


np.int64(16113)

In [ ]:
#Drop the NaN values
df = df.dropna(subset=['minimum nights'])

#We will go back to fix the bin again, but for now we will check the ANOVA test for minimum nights

In [49]:
#Create variable for each group's review ratings
m_short = df[df['minimum nights'] == 'Short stay']['review rate number']
m_medium = df[df['minimum nights'] == 'Medium stay']['review rate number']
m_monthly = df[df['minimum nights'] == 'Monthly stay']['review rate number']
m_long_term = df[df['minimum nights'] == 'Long term']['review rate number']


In [50]:
#Run ANOVA test to compare the means of the review ratings across the different room types
f_stat, p_value = stats.f_oneway(m_short, m_medium, m_monthly, m_long_term)
print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_value:.4f}")

F-statistic: 8.1367
P-value: 0.0000


Similarly, since p < 0.00001, this is also the most significant result across all factors

=> Need to test Turkey HSD to actually test this difference

In [51]:
#Run Turkey HSD post-hoc test to see which groups differ
tukey_nights = pairwise_tukeyhsd(df['review rate number'], df['minimum nights'], alpha=0.05)
print(tukey_nights)

      Multiple Comparison of Means - Tukey HSD, FWER=0.05       
   group1       group2    meandiff p-adj   lower   upper  reject
----------------------------------------------------------------
   Long term  Medium stay   0.0039 0.9997 -0.1066  0.1144  False
   Long term Monthly stay   0.0815 0.2315 -0.0292  0.1922  False
   Long term   Short stay    0.016 0.9812 -0.0918  0.1238  False
 Medium stay Monthly stay   0.0776 0.0001  0.0322  0.1231   True
 Medium stay   Short stay   0.0121 0.8444 -0.0258    0.05  False
Monthly stay   Short stay  -0.0655 0.0001  -0.104 -0.0271   True
----------------------------------------------------------------


* Tukey's HSD: Two significant pairs identified
  * Medium stay vs Monthly stay (p = 0.0001) 
  => Monthly stay rates significantly higher
  * Monthly stay vs Short stay (p = 0.0001) 
  => Monthly stay rates significantly higher

**Conclusion:** 

Monthly stays (8-30 nights) achieve significantly higher review ratings than both short-term and medium-term stays
=> This confirms the hypothesis that monthly guests are more invested in the space, hosts screen more carefully for longer commitments, while the host-guest relationship benefits from extended interaction.

**Product Recommendation:**
1. Promote monthly stay options as a premium listing tier with higher guest satisfaction
2. Provide hosts with tools to offer monthly stay discounts to attract longer-term committed guests
3. Create a "Monthly Stay Specialist" badge for hosts who consistently deliver high-rated extended stays

Finally, we test the last remaining factor — host_identity_verified — which showed the smallest difference in EDA (0.001 points)

Given the negligible gap observed, we expect this to show no statistical significance, but also test for completeness.


# Host-identity-verified

In [52]:
#Run t-test to compare the means of the review ratings between hosts that verifiy and those that do not verify their identity
verified = df[df['host_identity_verified'] == 'verified']['review rate number']
unconfirmed = df[df['host_identity_verified'] == 'unconfirmed']['review rate number']

t_stat, p_value = stats.ttest_ind(verified, unconfirmed, equal_var=False)
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")


T-statistic: 0.4959
P-value: 0.6199


**Conclusion**

With p = 0.62 > 0.05, we can say host identity verification has no meaningful impact on review ratings

=> This confirms the EDA finding and suggests Airbnb should not prioritize verification status as a quality indicator

# Overall Conclusion

After conducting formal hypothesis testing across all six host-controlled factors, we land at 3 key insights:

**1. Availability is the strongest driver of review ratings.**
Contrast to our previous EDA finding, hosts with lower availability achieve significantly higher ratings, confirming that selective hosting and investing in quality over volume leads to better guest experiences. 

=> Airbnb should encourage hosts to schedule maintenance breaks rather than listing year-round.

**2. Monthly stays hit the sweet spot for guest satisfaction.**
Similarly, listings with 8-30 night minimum stays significantly outperform short and medium stays. The extended host-guest relationship and higher commitment from both parties drives better experiences. 

=> Airbnb should promote monthly stays as a premium tier.

**3. Very high pricing creates an expectation gap that hurts ratings.**
As expected from EDA testing, very-high-priced listings ($1000+) rate significantly lower than medium and high-priced listings. Guests paying top dollar expect superior experiences, and when reality falls short, ratings suffer. 

=> Airbnb should provide pricing guidance tools that help hosts find their optimal price-to-quality ratio.

* Meanwhile, Host Verification showed no statistically significant impact on review ratings.

=> This suggests Airbnb should redirect product development resources away from these features and toward the the 3 high-impact areas identified above.